# 06 · Lightning — 1×N GPU

`TorchDistributor(num_processes=NUM_GPUS_PER_NODE, local_mode=True).run(td_lit_fit, ...)`로 child를 띄우고 각 child에서 `Trainer(devices=NUM_GPUS_PER_NODE, num_nodes=1, strategy='ddp')`로 학습. fit 본체는 패키지의 `recommender_pkg.lightning_trainer.fit`.

> 1×1은 [`05-launch_lightning_trainer_1x1.ipynb`](05-launch_lightning_trainer_1x1.ipynb) · M×N은 [`07-launch_lightning_trainer_MxN.ipynb`](07-launch_lightning_trainer_MxN.ipynb)

**클러스터**: Single node, DBR 17.3 LTS ML, Driver `g5.12xlarge` (4× A10G), Workers 0.

In [ ]:
%run ./00-setup

## 학습 함수 import 확인

In [ ]:
import os
import recommender_pkg
print(f"recommender_pkg v{recommender_pkg.__version__}")

In [ ]:
def td_lit_fit(**kwargs):
    """TorchDistributor child에 보낼 Lightning fit() thin wrapper. inline 함수로
    by-value pickling. 패키지가 설치되어 있어 sys.path 보강은 불필요."""
    from recommender_pkg.lightning_trainer import fit
    return fit(**kwargs)

## 1×N GPU

In [ ]:
import mlflow
from pyspark.ml.torch.distributor import TorchDistributor

NUM_GPUS_PER_NODE = 4  # 클러스터 driver의 GPU 수

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="lit-1xN", log_system_metrics=True) as run:
    run_id = run.info.run_id

TorchDistributor(
    num_processes=NUM_GPUS_PER_NODE,
    local_mode=True,
    use_gpu=True,
).run(
    td_lit_fit,
    experiment_path=EXPERIMENT_PATH,
    run_id=run_id,
    db_host=DB_HOST,
    db_token=DB_TOKEN,
    data_dir=DATA_DIR,
    ckpt_dir=os.path.join(CKPT_DIR, "lit-1xN"),
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    emb_dim=cfg["emb_dim"],
    tower_hidden=cfg["tower_hidden"],
    batch_size=cfg["batch_size_per_gpu"],
    num_epochs=cfg["num_epochs"],
    max_steps_per_epoch=cfg["max_steps_per_epoch"],
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    devices=NUM_GPUS_PER_NODE,
    num_nodes=1,
    topology="1xN",
    script_dir="",
)